# Prática 1 — Feature Extraction em CIFAR-10 (Versão Aluno)
**ENG4502 — Introdução à Ciência de Dados · PUC-Rio**

Neste notebook você irá implementar o conceito de **Feature Extraction** utilizando uma rede convolucional profunda pré-treinada (**ResNet-18**):
1. Congelar todos os parâmetros convolucionais do modelo (backbone).
2. Substituir a camada final (totalmente conectada) por uma nova linear adaptada ao CIFAR-10.
3. Treinar apenas a nova camada usando SGD.
4. Avaliar a performance do modelo.

Complete as seções marcadas com `### SEU CÓDIGO AQUI ###`.

## 0. Imports e Configuração de Hardware

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import Subset
import numpy as np
import matplotlib.pyplot as plt
import time

# Configurar dispositivo (GPU se disponível, senão CPU)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando o dispositivo: {device}')

## 1. Carregamento dos Dados com Subset Balanceado

O CIFAR-10 tem 50.000 imagens de treino. Para que a prática caiba no tempo de aula rodando em CPU, vamos extrair um **subset balanceado com 500 imagens por classe** (total de 5.000 imagens para treino e 1.000 para validação).

In [ ]:
N_TRAIN = 5000
N_VAL = 1000

# Pipelines de transformação (ResNet-18 exige 224x224 e normalização do ImageNet)
transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_full = datasets.CIFAR10(root='data', train=True, download=True, transform=transform)
val_full = datasets.CIFAR10(root='data', train=False, download=True, transform=transform)

def extract_balanced_subset(dataset, n_total):
    if n_total is None:
        return dataset
    n_per_class = n_total // 10
    indices = []
    class_counts = {c: 0 for c in range(10)}
    for idx, (_, label) in enumerate(dataset):
        if class_counts[label] < n_per_class:
            indices.append(idx)
            class_counts[label] += 1
        if len(indices) == n_total:
            break
    return Subset(dataset, indices)

train_dataset = extract_balanced_subset(train_full, N_TRAIN)
val_dataset = extract_balanced_subset(val_full, N_VAL)

# ========================================== 
# EXERCÍCIO 1: Crie os DataLoaders para train_dataset e val_dataset
# Dica: use batch_size=64 e num_workers=0 (essencial para Windows)
# ==========================================
### SEU CÓDIGO AQUI ###
train_loader = None
val_loader = None

print(f'Amostras de Treino: {len(train_dataset)} | Validação: {len(val_dataset)}')

## 2. Preparação do Modelo (Feature Extraction)

Você deve carregar a **ResNet-18**, congelar os seus parâmetros convolucionais do backbone e substituir a camada linear de classificação final.

In [ ]:
# ==========================================
# EXERCÍCIO 2: Configure a ResNet-18 para Feature Extraction
# 1. Carregue o modelo ResNet-18 pré-treinado com weights=models.ResNet18_Weights.IMAGENET1K_V1
# 2. Percorra model.parameters() congelando-os (requires_grad = False)
# 3. Substitua model.fc por uma nova nn.Linear adaptada para 10 classes
# ==========================================
### SEU CÓDIGO AQUI ###
model = None


model = model.to(device)

# Contar parâmetros treináveis vs. congelados para validar o congelamento
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total de Parâmetros: {total_params:,}')
print(f'Parâmetros Treináveis: {trainable_params:,} (deve ser exatamente 5,130)')

## 3. Função de Treinamento e Avaliação

In [ ]:
# ==========================================
# EXERCÍCIO 3: Defina a Loss e o Otimizador
# Dica: Garanta que o SGD receba para otimização APENAS os parâmetros da fc (model.fc.parameters())
# Use learning rate lr=0.01 e momentum=0.9
# ==========================================
### SEU CÓDIGO AQUI ###
criterion = None
optimizer = None

def train_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
    return running_loss / len(loader.dataset)

def evaluate(model, loader, device):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

## 4. Execução do Experimento

Rode por 5 épocas e observe o comportamento da Loss e Acurácia.

In [ ]:
num_epochs = 5
print('=== Iniciando Treinamento (Feature Extraction) ===')
t_start = time.time()

for epoch in range(1, num_epochs + 1):
    t0 = time.time()
    loss = train_epoch(model, train_loader, criterion, optimizer, device)
    acc = evaluate(model, val_loader, device)
    elapsed = time.time() - t0
    print(f'Época {epoch}/{num_epochs} | Loss: {loss:.4f} | Val Acc: {acc*100:.2f}% | Tempo: {elapsed:.1f}s')

total_time = time.time() - t_start
print(f'Treinamento completo em {total_time/60:.1f} minutos.')

## 5. Questões de Reflexão

1. Por que a acurácia de validação já começa alta desde a primeira época em comparação a treinar uma rede do zero?
2. O que aconteceria com os pesos congelados se tivéssemos passado `model.parameters()` (todos os parâmetros) para o otimizador SGD em vez de `model.fc.parameters()`?